In [ ]:
%%capture
!pip install transformers datasets peft accelerate bitsandbytes evaluate jiwer trl

In [ ]:
#final_data check
from datasets.load import load_dataset

dataset = load_dataset("abhi8799/Punjabi-STT-Small-8H")

README.md:   0%|          | 0.00/502 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/371M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/157M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/840 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/361 [00:00<?, ? examples/s]

In [ ]:
from datasets import Audio

# 1. Critical Step: Resample audio to 16kHz
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

In [ ]:
dataset = dataset.remove_columns(['id'])

In [ ]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

#Set the model
model_id = "openai/whisper-small"

# Load preprocessors
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_id)
# Set language to punjabi and task to transcribe
tokenizer = WhisperTokenizer.from_pretrained(model_id, language="punjabi", task="transcribe")
processor = WhisperProcessor.from_pretrained(model_id, language="punjabi", task="transcribe")

max_label_length = 448

def prepare_dataset_batched(batch):
    # Process audio features
    input_features = [
        feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
        for audio in batch["audio"]
    ]

    # CRITICAL: Tokenize WITHOUT truncation so we can see the true token length
    labels = tokenizer(batch["sentence"]).input_ids

    valid_input_features = []
    valid_labels = []

    # Iterate and completely DROP any pair where text exceeds 448
    for feature, label in zip(input_features, labels):
        if len(label) <= max_label_length:
            valid_input_features.append(feature)
            valid_labels.append(label)
        # If it's over 448, we skip it entirely to protect alignment!

    return {
        "input_features": valid_input_features,
        "labels": valid_labels
    }

# Apply to your dataset
dataset = dataset.map(
    prepare_dataset_batched,
    batched=True,
    batch_size=100,
    remove_columns=dataset["train"].column_names,
    num_proc=4
)

#ALready split: 840 + 361
# Split into train and test sets (e.g., 90% train, 10% eval)
# dataset = dataset["train"].train_test_split(test_size=0.1)

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

Map (num_proc=4):   0%|          | 0/840 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/361 [00:00<?, ? examples/s]

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Split inputs and labels since they need different padding methods
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # If bos token is appended in previous tokenization step, cut it
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [ ]:
from transformers import WhisperForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch


model_id = "openai/whisper-small"

# 1. Define the 8-bit quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True
)

# 2. Pass the config to the model initialization
model = WhisperForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quantization_config,  # Use this instead of load_in_8bit=True
    device_map="auto"
)

# Prepare model for PEFT int8 training quirks
model = prepare_model_for_kbit_training(model)

# Define LoRA Configuration
config = LoraConfig(
    r=32,                  # Rank of the adaptation
    lora_alpha=64,         # Scaling factor
    target_modules=["q_proj", "v_proj"], # Target the attention projection layers
    lora_dropout=0.05,
    bias="none"
)

model = get_peft_model(model, config)
model.print_trainable_parameters()
# You will notice that < 2% of the parameters are actually trainable now!

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

trainable params: 3,538,944 || all params: 245,273,856 || trainable%: 1.4429


In [ ]:
#FOR COLAB
# from google.colab import userdata
# HF_WRITE = userdata.get('<HF WRITE TOKEN>')

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

from transformers import Seq2SeqTrainingArguments

from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-gurmukhi-lora",  # This will also be your HF repo name
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    warmup_steps=100,
    max_steps=1000,

    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    predict_with_generate=False,
    logging_steps=25,

    fp16=True,
    gradient_checkpointing=True,
    optim="adamw_bnb_8bit",
    report_to=["tensorboard"],
    remove_unused_columns=False,

    # --- HUB SETTINGS ---
    push_to_hub=True,                    # Enable uploading to HF Hub
    hub_model_id="<HF_USERNAME>/whisper-small-gurmukhi-lora", # Target repo ID
    hub_strategy="every_save",           # Upload checkpoints as they are saved
    hub_token= HF_WRITE
)

# Fix configurations for generating text during evaluation
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    processing_class=processor,
)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss,Validation Loss


In [ ]:
# 1. Start training
trainer.train()

# 2. Push the final trained LoRA adapters and tokenizer configurations to the Hub
trainer.push_to_hub(tags="automatic-speech-recognition", commitment_message="End of fine-tuning on Gurmukhi dataset")